# The QAOA part of the craiceann problem

**Objective:** Verify the software stack from Data Ingestion -> Hamiltonian -> Circuit Transpilation.

**Context:** This notebook takes "stub" physics data (non-interacting cracks) and tests if the Qiskit pipeline can successfully generate a hardware-compatible QAOA circuit.



In [13]:
import numpy as np
import rustworkx as rx

from qiskit.quantum_info import SparsePauliOp
from qiskit.circuit.library import QAOAAnsatz
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

from qiskit_ibm_runtime.fake_provider import FakeTorino

## 1. Data Ingestion & Normalization
We import raw edge data.

**Critical Fix:** Raw physics weights vary by 20 orders of magnitude. We apply Min-Max normalization to compress them into a range suitable for the COBYLA optimizer $[0.1, 1.0]$.

In [14]:
# MOCK DATA: from Step-3.ipynb (Linear Graph: 0-1-2)
# Structure: (Node_U, Node_V, Raw_Weight_G)
raw_graph_data = [
    (0, 1, 1.555e-10), # Small crack
    (1, 2, 3.111e-10)  # Larger crack 
]

print("--- 1. Data Ingestion ---")
weights = np.array([w for u, v, w in raw_graph_data])

# Normalize weights to [0.1, 1.0] for stable QAOA convergence
w_min, w_max = weights.min(), weights.max()

# Safety check for uniform weights
if w_max == w_min:
    norm_weights = np.ones_like(weights) * 1.0
else:
    norm_weights = 0.1 + (weights - w_min) * (0.9 / (w_max - w_min))

print(f"Raw Weights (Physical G): {weights}")
print(f"Normalized Weights (Quantum J): {norm_weights}")

# Rebuild Graph using rustworkx
G_quantum = rx.PyGraph()

# 1. Add Nodes (indices 0, 1, 2)
# We determine the number of nodes needed
max_node_idx = max(max(u, v) for u, v, _ in raw_graph_data)
G_quantum.add_nodes_from(range(max_node_idx + 1))

# 2. Add Edges with Normalized Weights as payload
for i, (u, v, _) in enumerate(raw_graph_data):
    # add_edge(parent, child, edge_weight/payload)
    G_quantum.add_edge(u, v, norm_weights[i])

print(f"Graph built: {G_quantum.num_nodes()} nodes, {G_quantum.num_edges()} edges")

--- 1. Data Ingestion ---
Raw Weights (Physical G): [1.555e-10 3.111e-10]
Normalized Weights (Quantum J): [0.1 1. ]
Graph built: 3 nodes, 2 edges


## 2. Hamiltonian Construction
We map the Weighted Max-Cut problem to the Ising Hamiltonian:
$$H_C = \sum_{(i,j) \in E} w_{ij} Z_i Z_j$$

*Note: Qiskit uses Little-Endian ordering (qubit 0 is the right-most character).*

In [ ]:
print(f"\n--- 2. Hamiltonian Construction ---")

pauli_list = []
num_qubits = G_quantum.num_nodes()

# rustworkx iteration: edge_index_map().values() returns (u, v, payload)
for u, v, weight in G_quantum.edge_index_map().values():
    
    # Initialize identity string
    op_str = ['I'] * num_qubits
    
    # Assign Pauli Z to the interacting nodes
    op_str[u] = 'Z'
    op_str[v] = 'Z'
    
    # Reverse string for Little-Endian
    pauli_str = "".join(op_str[::-1])
    
    pauli_list.append((pauli_str, weight))

hamiltonian = SparsePauliOp.from_list(pauli_list)
print(f"Ising Hamiltonian Terms: \n{hamiltonian}")


--- 2. Hamiltonian Construction ---
Ising Hamiltonian Terms: 
SparsePauliOp(['IZZ', 'ZZI'],
              coeffs=[0.1+0.j, 1. +0.j])


## 3. QAOA Circuit Generation
We construct the Variational Quantum Circuit using the standard `QAOAAnsatz`.

In [16]:
print(f"\n--- 3. QAOA Circuit Generation (p=1) ---")

qaoa_circuit = QAOAAnsatz(cost_operator=hamiltonian, reps=1)
qaoa_circuit.measure_all()

print(f"Logical Circuit Depth: {qaoa_circuit.depth()}")
print(f"Logical Qubits: {qaoa_circuit.num_qubits}")


--- 3. QAOA Circuit Generation (p=1) ---
Logical Circuit Depth: 2
Logical Qubits: 3


## 4. Transpilation & Hardware Simulation
We simulate mapping this circuit to **IBM Torino** (Heavy-Hex Topology).

In [17]:
print(f"\n--- 4. Hardware Transpilation (Simulation) ---")

# Load Fake Backend (Simulates IBM Heron processor)
backend = FakeTorino()
print(f"Target Backend: {backend.name} ({backend.num_qubits} qubits)")

# Generate Pass Manager (Optimization Level 3 for best routing)
pm = generate_preset_pass_manager(backend=backend, optimization_level=3)

# Transpile
transpiled_circuit = pm.run(qaoa_circuit)

# Analysis
phys_depth = transpiled_circuit.depth()
ops = transpiled_circuit.count_ops()
swap_count = ops.get('swap', 0)

print(f"Physical Circuit Depth: {phys_depth}")
print(f"Total Gate Operations: {ops}")
print(f"SWAP Gates Inserted: {swap_count}")

if swap_count > 0:
    print(">> WARNING: Topology mismatch detected. SWAP overhead added.")
else:
    print(">> SUCCESS: Graph fits natively on coupling map (Linear topology matches).")


--- 4. Hardware Transpilation (Simulation) ---
Target Backend: fake_torino (133 qubits)
Physical Circuit Depth: 25
Total Gate Operations: OrderedDict({'rz': 16, 'sx': 11, 'cz': 4, 'measure': 3, 'barrier': 1})
SWAP Gates Inserted: 0
>> SUCCESS: Graph fits natively on coupling map (Linear topology matches).
